# Enkaz — training notebook

Pipeline: **setup → data → preparation → training → evaluation**. Heavy logic lives in the `enkaz`
package; this notebook only calls it. Every stage writes a marker to Google Drive
(`MyDrive/enkaz/stages`) and is skipped when re-run, so a disconnected session can resume.

Raw downloads go to the Colab disk (`/content/enkaz_work`, re-downloadable). Catalogues,
processed data and checkpoints go to Drive.

Imagery: Maxar Open Data Program, CC BY-NC 4.0. Labels: BRIGHT (Chen et al., 2025), derived from
Copernicus EMS. Reference gradings: © European Union, Copernicus EMS (EMSR648).

## 0. Setup

In [ ]:
# 0.1 Mount Google Drive (persistent storage for markers, catalogues and checkpoints)
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 0.2 Get the code and install it.
# CODE_SOURCE = "github": clone the repo (a read-only token is asked for if the repo is private).
# CODE_SOURCE = "upload": the repo folder was uploaded to /content/enkaz with "Upload to Colab".
import os
import subprocess
import sys

CODE_SOURCE = "github"
REPO = "github.com/emrtunali/enkaz"
CODE_DIR = "/content/enkaz"

if CODE_SOURCE == "github":
    if os.path.isdir(f"{CODE_DIR}/.git"):
        subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
    else:
        from getpass import getpass

        token = getpass("GitHub read-only token (leave empty if the repo is public): ").strip()
        auth = f"x-access-token:{token}@" if token else ""
        subprocess.run(
            ["git", "clone", "--depth", "1", f"https://{auth}{REPO}.git", CODE_DIR], check=True
        )
        del token, auth
assert os.path.isdir(f"{CODE_DIR}/src/enkaz"), "code not found"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{CODE_DIR}[geo,train]"], check=True
)
subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--format=%h %s"], check=False)

In [ ]:
# 0.3 Environment and storage
import importlib
import json
import shutil
import subprocess

import enkaz
from enkaz import config
from enkaz.stages import StageRunner

importlib.reload(config)
paths = config.default_paths().ensure()
runner = StageRunner(paths.markers)

print("enkaz", enkaz.__version__)
print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout.strip()
    or "no GPU"
)
print(f"RAM {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9:.1f} GB")
for p in (paths.work, paths.persist):
    du = shutil.disk_usage(p)
    print(f"{p}: {du.free / 1e9:.0f} GB free of {du.total / 1e9:.0f} GB")

## 1. Data catalogues

Small, fast stages (a few minutes in total). Output is written to `MyDrive/enkaz/catalog`.

In [ ]:
# 1.1 Maxar Open Data: crawl the STAC catalogue of the event (metadata only, ~2,100 tiles, ~3 min)
from enkaz.data import maxar


def stage_maxar():
    rows = maxar.crawl_event(config.MAXAR_EVENT_COLLECTION)
    (paths.catalog / "maxar_items.json").write_text(json.dumps(rows))
    pre = sum(maxar.phase(r, config.EVENT_DATE) == "pre" for r in rows)
    return {"items": len(rows), "pre": pre, "post": len(rows) - pre}


print(runner.run("maxar_catalog", stage_maxar))
maxar_items = json.loads((paths.catalog / "maxar_items.json").read_text())

In [ ]:
# 1.2 The 81 provinces: what open data exists for each one?
from collections import Counter

from enkaz.data import provinces as prov


def stage_provinces():
    path = prov.fetch_adm1(config.GEOBOUNDARIES_ADM1_API, paths.catalog)
    p = prov.Provinces(path)
    table = prov.coverage_table(p, maxar_items, config.EVENT_DATE, config.CITY_CENTERS)
    (paths.catalog / "province_coverage.json").write_text(
        json.dumps(table, indent=1, ensure_ascii=False)
    )
    return {"provinces": len(p), "status": dict(Counter(r["status"] for r in table))}


print(runner.run("province_coverage", stage_provinces))
coverage = json.loads((paths.catalog / "province_coverage.json").read_text())
for r in coverage:
    if r["status"] != prov.STATUS_NONE:
        print(
            f"{r['province']:16s} {r['status']:24s} pre {r['pre_tiles']:4d}  "
            f"post {r['post_tiles']:4d}  {r['labeled_cities']}"
        )
print(
    sum(r["status"] == prov.STATUS_NONE for r in coverage),
    "provinces without post-event VHR imagery (pipeline ready, no data to run on)",
)

In [ ]:
# 1.3 Copernicus EMS EMSR648 grading vectors (36 packages, ~95 MB) -> Drive
from enkaz.data import cems


def stage_cems():
    out = cems.download_all(config.CEMS_ACTIVATION_PAGE, paths.persist / "raw_cems")
    for o in out:
        o.update(cems.summarize_builtup(o["folder"]))
    (paths.catalog / "cems_products.json").write_text(json.dumps(out, indent=1, ensure_ascii=False))
    return {"packages": len(out)}


print(runner.run("cems_vectors", stage_cems))
for o in json.loads((paths.catalog / "cems_products.json").read_text()):
    if o["product"] == "PRODUCT":
        print(f"AOI{o['aoi']:02d} {str(o['locality']):14s} {o['grades']}")

## 2. BRIGHT labels and pre-event images (Türkiye only)

Fetched member-by-member from the multi-event archives with HTTP range requests:
~1.2 GB of label masks and ~3.5 GB of pre-event images, **15–25 min**. Files go to the Colab
disk; re-running after a disconnect only fetches what is missing. The tile index is saved to
Drive.

In [ ]:
# 2.1 Download BRIGHT Türkiye members (idempotent; progress every 100 files)
from enkaz.data import bright
from enkaz.data.remotezip import RemoteZip

bright_dir = paths.raw / "bright"
for kind, archive in bright.ARCHIVES.items():
    rz = RemoteZip.from_url(f"{config.BRIGHT_BASE}/{archive}")
    entries = bright.members(rz)
    print(f"{kind}: {len(entries)} Türkiye tiles")
    bright.fetch_members(rz, entries, bright_dir / kind)

In [ ]:
# 2.2 Tile index: georeference, city and class pixel counts of every label tile (~2 min)
def stage_bright_index():
    rows = bright.tile_index(bright_dir / "target", config.CITY_CENTERS)
    (paths.catalog / "bright_tiles.json").write_text(json.dumps(rows))
    return {"tiles": len(rows), "cities": dict(Counter(r["city"] for r in rows))}


print(runner.run("bright_index", stage_bright_index))
tiles = json.loads((paths.catalog / "bright_tiles.json").read_text())
by_city = {}
for t in tiles:
    c = by_city.setdefault(t["city"], Counter())
    c.update({k: t[k] for k in ("px_intact", "px_damaged", "px_destroyed")}, n=1)
for city, c in sorted(by_city.items(), key=lambda kv: str(kv[0])):
    print(
        f"{str(city):14s} tiles {c['n']:4d}  intact {c['px_intact'] / 1e6:6.1f}M  "
        f"damaged {c['px_damaged'] / 1e6:5.2f}M  destroyed {c['px_destroyed'] / 1e6:5.2f}M"
    )